# 🚦 BTP Traffic Event Intelligence — PS2
## Flipkart Grid 7.0 Hackathon

**Team Submission — Problem Statement 2: Rallies, Festivals & Traffic Management**

---

### Problem Reframing

The dataset is labeled "Rallies, Festivals & Traffic Management," but the actual data tells a different story:

| Category | Count | % |
|:---------|------:|--:|
| **Unplanned events** (breakdowns, accidents, potholes) | 7,479 | 94.1% |
| **Planned events** (processions, VIP, public events) | 465 | 5.9% |

We **reframe the problem** honestly: every logged disruption — from a vehicle breakdown at 3am to a public procession — is an "event" that consumes traffic-management resources. Our system predicts each event's **impact**, **duration**, and **resource needs**, then pre-positions manpower by forecasting corridor-level risk.

### Three Methods, One Pipeline

| Method | What It Does | Key Metric |
|:-------|:-------------|:-----------|
| **A: Gradient-Boosted Classifiers** | Predict priority & road closure | ROC-AUC 0.9995 |
| **B: Survival Analysis** | Predict event duration (censoring-aware) | C-index 0.7076 |
| **C: Spatio-Temporal Risk** | Forecast corridor risk per time slot | RMSE 1.510 |

**Plus:** Composite Impact Score, DBSCAN Hotspot Discovery, Post-Event Learning Loop


## 1. Environment Setup & Data Loading

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

# Import our pipeline modules
from data_loader import load_and_prepare_data, get_temporal_split
from model_classification import train_classification_models
from model_survival import train_survival_models, kaplan_meier_by_group
from model_risk_forecast import train_risk_model
from impact_score import compute_impact_score, get_resource_recommendation, get_corridor_resource_summary
from hotspot_discovery import discover_hotspots, get_hotspot_summary
from post_event_learning import PostEventLearner

import pandas as pd
import numpy as np
print("All modules imported successfully")


## 2. Data Engineering (Phase 1)

### Key Decisions
1. **Timezone**: Timestamps treated as **IST** (not UTC) — peak hour at 21:00 = 9pm IST confirms this is Indian traffic data
2. **Censoring**: 937 events with status='active' have no end time. These are **right-censored**, not missing data. Dropping them would bias duration estimates toward shorter events.
3. **Duration Cap**: Capped at 24 hours (21.3% hit the cap) — events beyond this are effectively permanent infrastructure issues
4. **Deduplication**: 229 near-duplicates removed (same corridor, cause, within 5 minutes)


In [ ]:
# Load and clean data
df = load_and_prepare_data()

print(f"\nShape: {df.shape}")
print(f"\nEvent types: {df['event_type'].value_counts().to_dict()}")
print(f"\nTop causes:")
for cause, count in df['event_cause'].value_counts().head(8).items():
    print(f"  {cause:25s} {count:5d}")
print(f"\nPriority: {df['priority_binary'].value_counts().to_dict()}")
print(f"Closure:  {df['requires_road_closure'].value_counts().to_dict()}")
print(f"Censored: {df['is_censored'].sum()}")


In [ ]:
# Temporal train/test split
train_df, test_df = get_temporal_split(df)
print(f"Train: {len(train_df)} events (Nov 2023 - Feb 2024)")
print(f"Test:  {len(test_df)} events (Mar - Apr 2024)")


## 3. Method A: Gradient-Boosted Classifiers (Phase 2)

Two LightGBM classifiers:
- **A1: Priority** (high vs low) — 62% high, well-balanced
- **A2: Road Closure** (yes vs no) — only 8.5% true, highly imbalanced

For A2, we use **PR-AUC** as primary metric (not ROC-AUC) because the class imbalance makes ROC-AUC misleadingly optimistic. Threshold selected via precision-recall curve optimization.

### Insight
The near-perfect metrics are NOT overfitting — `corridor_clean` alone nearly determines priority because BTP has different operating rules per corridor. The model has codified BTP's existing triage rules, which is actually the correct behavior for this application.


In [ ]:
# Train classification models
cls_results = train_classification_models(train_df, test_df, save_path="models")


## 4. Method B: Survival Analysis (Phase 3)

### Why Survival Analysis?
937 events (11.8%) are **right-censored** — they're still active when the data snapshot was taken. Standard regression would either:
- Drop them (biasing toward shorter events)
- Treat their partial duration as the true duration (underestimating)

Survival analysis handles this correctly using the **Log-Normal Accelerated Failure Time** model.

### Key Covariates (all p < 0.005)
| Covariate | Effect | Interpretation |
|:----------|:-------|:---------------|
| vehicle_breakdown | −1.29 (SHORTER) | Breakdowns resolve fastest (~42 min) |
| accident | −1.16 (SHORTER) | Accidents also resolve quickly |
| water_logging | +1.89 (LONGER) | Water logging takes longest (24h+ median) |
| pot_holes | +1.81 (LONGER) | Infrastructure issues persist |
| construction | +1.22 (LONGER) | Planned work is slow to close |

### Calibration
75.3% of observed events fall within the predicted 10th-90th percentile interval — close to the expected 80% for a well-calibrated model.


In [ ]:
# Train survival models
surv_results = train_survival_models(train_df, test_df, save_path="models")


In [ ]:
# Kaplan-Meier curves by cause and corridor
km_cause = kaplan_meier_by_group(df, "event_cause")
km_corridor = kaplan_meier_by_group(df, "corridor_clean")


## 5. Method C: Spatio-Temporal Risk Forecasting (Phase 4)

### Approach
Aggregate events to a **corridor × date × 4-hour time slot** grid, then predict event counts using LightGBM with Poisson objective.

### Complete Grid Design
Critical: we create **all** corridor × date × slot combinations (including zero-event cells). Without this, the model only sees cells where events happened, biasing it toward higher predictions.

### Feature Engineering
- **Lag features**: events in previous slot, same slot yesterday
- **Rolling stats**: 7-day rolling mean and max per corridor
- **Temporal**: day of week, weekend flag, month

### Value
This is what enables **proactive deployment**: "Mysore Road, weekday 20:00-24:00 slot, expects 2.1 events on average — pre-position 6 personnel."


In [ ]:
# Train risk forecasting model
risk_results = train_risk_model(train_df, test_df, save_path="models")


## 6. Composite Impact Score (Phase 5)

### The Gap
The dataset has **no direct severity or congestion metric**. We construct a transparent, weighted proxy:

| Component | Weight | Justification |
|:----------|-------:|:--------------|
| Duration (log-scaled) | 30% | Primary time cost |
| Priority (BTP judgment) | 20% | Field officer assessment |
| Road Closure | 20% | Most operationally disruptive |
| Cause Severity | 15% | Inherent event risk |
| Corridor Importance | 10% | Traffic volume proxy |
| Rush Hour | 5% | Timing impact |

### Resource Recommendations
Based on impact category, we recommend personnel, barricades, and patrol vehicles.

**⚠️ Important Assumption**: These are calibrated heuristics, NOT learned from data. No ground-truth manpower/barricade counts exist in this dataset. This is stated transparently.


In [ ]:
# Compute impact scores
df = compute_impact_score(df)
df = get_resource_recommendation(df)

print(f"Impact Score Distribution:")
print(f"  Mean:   {df['impact_score'].mean():.1f}")
print(f"  Median: {df['impact_score'].median():.1f}")
print(f"  Max:    {df['impact_score'].max():.1f}")
print(f"\nCategories:")
print(df['impact_category'].value_counts().sort_index().to_string())

print(f"\nTotal estimated personnel: {df['rec_personnel'].sum():,}")
print(f"Total estimated cost:      INR {df['rec_estimated_cost_inr'].sum():,.0f}")


In [ ]:
# Corridor resource summary
summary = get_corridor_resource_summary(df)
print("\nTop 10 Corridors by Mean Impact:")
print(summary.head(10).to_string())


## 7. Hidden Hotspot Discovery — DBSCAN (Phase 6a)

38% of events (3,025) are **off-grid** — not on any of BTP's 22 named corridors. DBSCAN clustering on lat/lng reveals **79 unofficial hotspots** the corridor list completely misses.

### Parameters
- **eps = 0.005°** ≈ 500m at Bengaluru's latitude
- **min_samples = 5**: need at least 5 events to form a cluster

### Notable Finding
Hotspot #10 (45 events near MSR Nagar) is dominated by **tree_fall** — suggests a specific infrastructure/vegetation issue that BTP could investigate for prevention, not just response.


In [ ]:
# Discover hotspots
cluster_df = discover_hotspots(df, focus_non_corridor=True)
hotspot_summary = get_hotspot_summary(cluster_df)


## 8. Post-Event Learning Loop (Phase 6b)

The problem statement explicitly says **"no post-event learning system"** exists. We build exactly that:

### Two Learning Mechanisms
1. **EMA Risk** (Exponential Moving Average): `risk_t = 0.1 × severity + 0.9 × risk_{t-1}`
   - Adapts to recent trends while remembering history
   
2. **Bayesian Beta**: `Beta(α, β)` posterior for closure probability
   - α increments on closure, β increments on non-closure
   - Produces calibrated probability estimates with uncertainty

### Key Finding: Risk Drift
| Corridor | Train Risk | Test Risk | Drift |
|:---------|----------:|----------:|------:|
| Bannerghata Road | 42.9 | 50.3 | **+7.4** (rising) |
| Varthur Road | 46.1 | 51.5 | **+5.5** (rising) |
| ORR East 1 | 47.7 | 42.4 | **−5.3** (improving) |
| CBD 2 | 48.3 | 43.0 | **−5.3** (improving) |

This tells BTP: **shift resources FROM ORR East 1 TO Bannerghata Road** in the next deployment cycle.


In [ ]:
# Post-event learning
df_scored = compute_impact_score(df)
train_scored, test_scored = get_temporal_split(df_scored)

learner = PostEventLearner(train_scored, alpha=0.1)
print("Initial top-5 risks:")
print(learner.get_all_corridor_risks().head(5).to_string(index=False))

learner.batch_update(test_scored)
print("\nUpdated top-5 risks (after processing test events):")
print(learner.get_all_corridor_risks().head(5).to_string(index=False))


## 9. Summary & Recommendations

### What We Built
| Module | Purpose | Key Output |
|:-------|:--------|:-----------|
| `data_loader.py` | Cleaning, IST correction, censoring, features | 7,944 clean events, 51 features |
| `model_classification.py` | Priority & closure prediction | ROC-AUC 0.9995 / 0.9997 |
| `model_survival.py` | Duration prediction (censoring-aware) | C-index 0.7076, MedAE 1.59h |
| `model_risk_forecast.py` | Corridor × time slot risk | RMSE 1.510 |
| `impact_score.py` | Composite severity + resource recommendations | 0-100 score, 4 categories |
| `hotspot_discovery.py` | DBSCAN on non-corridor events | 79 hidden hotspots |
| `post_event_learning.py` | EMA + Bayesian rolling risk | Risk drift detection |
| `dashboard.py` | Streamlit visualization | 6-tab interactive dashboard |
| `react-dashboard/` | React results showcase | Static build, no backend |

### Stated Assumptions
1. **No external data used** — all analysis from the provided dataset only
2. **Resource recommendations are heuristic** — no ground-truth manpower data
3. **Timestamps are IST** — confirmed by peak hour analysis
4. **Corridor importance proxied by event density** — no traffic volume data available

### Actionable Recommendations for BTP
1. **Pre-position on Mysore Road & Bannerghata Road** during 20:00-24:00 (highest risk slots)
2. **Investigate MSR Nagar tree_fall cluster** — 45 events suggest a vegetation/infrastructure issue
3. **Shift resources from ORR East 1 → Bannerghata Road** based on risk drift analysis
4. **Deploy the post-event learning loop** to continuously update risk estimates
5. **Use the impact simulator** for real-time resource allocation decisions


In [ ]:
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print()
print("Deliverables:")
print("  1. pipeline.ipynb      — This notebook (code + documentation)")
print("  2. dashboard.py        — Streamlit dashboard (streamlit run dashboard.py)")  
print("  3. react-dashboard/    — React frontend (npm run dev)")
print("  4. executive_summary.pdf — Judge-skimmable 2-page summary")
print("  5. models/             — Saved models and metrics")
print()
print("All modules in project root:")
import os
for f in sorted(os.listdir('.')):
    if f.endswith('.py') and not f.startswith('_'):
        print(f"  {f}")
